# Confusion Matrix Notebook

In this notebook we will create a confusion matrix for the behavioral responses across our glint experiment, for humans and models. 

Essentially we will create 18 x 18 matrix (familiar) and 4 x 4 matrix (novel), Each value in said matrix will correspond to the amount of times say someone gave the response cat for a cat target trial, dog trial, and so one so forth. This way we can see what are the responses participants give for each target object class.

Of importance, you need to go and ge


You will need to exclude participants 

## Import packages

In [1]:
# Import Necessary Packages 
import os
import pandas as pd
import glob
import numpy as np
import pdb

## Create a function to loop through each participants data and get all participants trials for each object class.

In [ ]:
'''
                    Exclusion Criteria 1: RTs less than or equal to 100ms, and counts the number of fast responses
                    Exclusion Criteria 2: RTs outside mean ± 2*SD
                    We also remove no responses first, so that we are only looking at RTs for trials where there was a response, 
                    and then we apply the RT filters to those trials. This way we can accurately count 
                    the number of no responses, fast responses, and RT outliers separately.'''

In [ ]:
base_columns = [
    "experiment_label",
    "id",
    "condition",
    "animacy",
    "target_image",
    "response_label",
    "correct (0=incorrect, 1=correct)",
]


'''
# Create empty lists to store summary dataframes for each object class
airplane_summary = []
apple_summary = []
avocado_summary = []
bear_summary = []
binoculars_summary = []
bread_summary = []
butterfly_summary = []
car_summary = []
cat_summary = []
corn_summary = []
cow_summary = []
dog_summary = []
house_summary = []
lawnmower_summary = []
mushroom_summary = []
pineapple_summary = []
snail_summary = []
teapot_summary = []
'''

In [ ]:
objects = ['airplane', 'apple', 'avocado', 'bear', 'binoculars', 'bread', 'butterfly', 'car', 'cat', 'corn', 'cow', 'dog', 'house', 'lawnmower', 'mushroom', 'pineapple', 'snail', 'teapot']

summaries = {}

for object in objects:
    # create a label for the object class dataframes
    label = f"{object}_summary"

    # add label to list of summary dataframes
    summaries[label] = []



In [2]:
import glob
import pandas as pd

experiment = 'familiar' # set experiment variable to either 'familiar' or 'novel', this will determine which data files we pull and how we structure the summary dataframes


# Define the paths 
base = '/zpool/vladlab/data_drive/glint_master/data/adult_data'
    
core_path = f'{base}/{experiment}_glint' # set core path using experiment variable and base path

# participants we excluded in our original analysis
excluded_participants = ['66afcaa60f7d8f58dc21db8e', '66cec5a3fdf1fe2c010e9971']

# List of object classes in the experiment, which we will loop through to create summary dataframes for each object class
objects = ['airplane', 'apple', 'avocado', 'bear', 'binoculars', 'bread', 'butterfly', 'car', 'cat', 'corn', 'cow', 'dog', 'house', 'lawnmower', 'mushroom', 'pineapple', 'snail', 'teapot']

# Initialize empty dataframe
participant_summary_df = pd.DataFrame()

sub_summary = pd.DataFrame()

base = '/zpool/vladlab/data_drive/glint_master/data/adult_data'
    
# import the files
files = glob.glob(f'{core_path}/*.csv') 
print(f"You have selected {experiment} glint experiment")


    
for obj in objects:
    for file in files:

        # Take the data for one participant at a time
        data = pd.read_csv(file)

        # Check if the participant is in the excluded list
        if data.loc[0, 'participant'] in excluded_participants:
            continue # skip this participant if they are in the excluded list
        

        '''        
        We apply the exclusion criteria to the data for each participant before we create the summary dataframes.
        Exclusion criteria 1: Remove No responses
        Exclusion criteria 2: Remove RTs less than or equal to 100ms
        Exclusion criteria 3: Remove RTs outside mean ± 2*SD
        
        '''
        # Exclusion criteria 1: Remove No responses
        data_exclusion_1 = data[data['response_label'] != 'no_response']

        # Exclusion criteria 2: Remove RTs less than or equal to 100ms
        data_exclusion_2 = data_exclusion_1[data_exclusion_1['key_resp.rt'] >= 0.1]

        # Exclusion criteria 3: Remove RTs outside mean ± 2*SD
        rt_mean = data_exclusion_2['key_resp.rt'].mean()
        rt_std = data_exclusion_2['key_resp.rt'].std()
        data_exclusion_3 = data_exclusion_2[(data_exclusion_2['key_resp.rt'] >= rt_mean - 2*rt_std) & (data_exclusion_2['key_resp.rt'] <= rt_mean + 2*rt_std)]



        # Filter for the current object class
        if experiment == 'familiar':
            object_trials = data_exclusion_3[data_exclusion_3['object_class'] == obj]
        
        
        # Extract only the columns you want
        if experiment == 'familiar':
            subset = object_trials[[
                'participant',
                'img_condition', # condition (natural, scrambled, line_drawing)
                'object_class', # The correct object class of the image they were shown (ex: airplane, car, etc.)
                'object_category', # The animacy category of the image they were shown (animate, inanimate natural, inanimate artificial)
                'response_label', # Response given by subject (ex: car, dog, cat, etc.)
                'response_animacy', # Animacy of the response they gave (animate, inanimate natural, inanimate artificial)
                'key_resp.corr', # Whether the response was correct (1) or incorrect (0)
                'key_resp.keys', # The actual key they pressed for their response
                'object_class', # The object class of the image they were shown (ex: airplane, car, etc.)
                'prompt_1', # The first prompt they saw for that trial (ex: "Is this a car?")
                'prompt_2', 
                'prompt_3', 
                'prompt_4', 
                'prompt_5', 
                'prompt_6', 
                'prompt_7', 
                'prompt_8', 
                'prompt_9']].copy()
            
        if experiment == 'novel':
            subset = object_trials[[
                'participant',
                'response_label', 
                'key_resp.corr', 
                'key_resp.keys', 
                'img_condition', 
                'img_class',
                'prompt_1',
                'prompt_2',
                'prompt_3',
                'prompt_4']].copy()
            
        column_map = {
            "reponse_label": "response_by_subject",
            "key_resp.corr": "correct (0=incorrect, 1=correct)",
            "key_resp.keys": "keyboard_response",
            "img_condition": "condition_block",
            "object_category": "correct_animacy",
            "img_class": "correct_object_class",
            "object_class": "correct_object_class"}

        #rename the colimns to standardize across experiment versions
        data = subset.rename(columns=column_map)

        
        # Append to the summary list
        participant_summary_df = pd.concat([participant_summary_df, data], ignore_index=True) #append the subset dataframe for this object class and participant to the overall participant summary dataframe



    # Combine all into one dataframe
    sub_summary = pd.concat([sub_summary, participant_summary_df], ignore_index=True)


You have selected familiar glint experiment


## Now that we have all participants trials with Exclusion Criteria implemented, let's look at the data

In [6]:
objects = sub_summary['correct_object_class']
for obj in objects:
    object_summary = sub_summary[sub_summary['correct_object_class'] == obj]
    summaries[f"{obj}_summary"] = object_summary

ValueError: cannot reindex on an axis with duplicate labels